In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from google.colab import drive

drive.mount('/content/drive')



In [ ]:
import numpy as np
import pandas as pd
from numba import njit, prange
from scipy.sparse import coo_matrix


dtypes = {"userId":"int32", "movieId":"int32", "rating":"float32"}
df = pd.read_csv("/content/drive/MyDrive/ratings.csv", usecols=['userId', 'movieId', 'rating'], dtype=dtypes)

# Vectorized mappings

unique_users, user_indices = np.unique(df['userId'].values, return_inverse=True)
unique_items, item_indices = np.unique(df['movieId'].values, return_inverse=True)
ratings = df['rating'].values

n_users = len(unique_users)
n_items = len(unique_items)

rng = np.random.default_rng(42)
random_values = rng.random(len(ratings))
train_mask = random_values < 0.8
test_mask = ~train_mask

train_matrix_user = coo_matrix((ratings[train_mask], (user_indices[train_mask], item_indices[train_mask])), shape=(n_users,n_items)).tocsr()
test_matrix_user = coo_matrix((ratings[test_mask], (user_indices[test_mask], item_indices[test_mask])), shape=(n_users,n_items)).tocsr()
train_matrix_item = coo_matrix((ratings[train_mask], (item_indices[train_mask], user_indices[train_mask])), shape=(n_items,n_users)).tocsr()

train_user_indptr = train_matrix_user.indptr
train_user_indices = train_matrix_user.indices
train_user_data = train_matrix_user.data

test_user_indptr = test_matrix_user.indptr
test_user_indices = test_matrix_user.indices
test_user_data = test_matrix_user.data

train_item_indptr = train_matrix_item.indptr
train_item_indices = train_matrix_item.indices
train_item_data = train_matrix_item.data

global_average = np.float32(np.mean(train_user_data))


In [ ]:
@njit(parallel=True, fastmath=True)
def als_algorithm(user_indptr, user_indices, user_data,
    item_indptr, item_indices, item_data,
    U, V, user_bias, item_bias, global_average,
    lam, gamma, tau, U_reg, V_reg, I, k):
	n_users = len(user_indptr) - 1
	n_items = len(item_indptr) - 1

	for m in prange(n_users):
		start_idx = user_indptr[m]
		end_idx = user_indptr[m+1]
		item_counter = end_idx - start_idx
		if item_counter == 0:
			user_bias[m] = 0
			continue

		numerator = 0.0
		for i in range(start_idx, end_idx):
			n = user_indices[i]
			r = user_data[i]

			dot_product = 0.0
			for j in range(k):
				dot_product += U[m,j] * V[n,j]
			numerator += lam * (r - global_average - item_bias[n] - dot_product)

		denominator = lam * item_counter + gamma
		user_bias[m] = numerator / denominator



	for m in prange(n_users):
		start_idx = user_indptr[m]
		end_idx = user_indptr[m+1]
		if start_idx == end_idx:
			U[m, :] = 0.0
			continue
		A = np.zeros((k, k), dtype=np.float32)
		b = np.zeros(k, dtype=np.float32)
		for i in range(start_idx, end_idx):
			n = user_indices[i]
			r = user_data[i]
			# Remove the biases before fitting latent factors.
			residual = (r - global_average - user_bias[m] - item_bias[n])
			for r_idx in range(k):
				V_r = V[n,r_idx]
				b[r_idx] += np.float32(lam * V_r * residual)
				for c_idx in range(k):
					A[r_idx,c_idx] += np.float32(lam * V_r * V[n,c_idx])
		# L2 regularization for User factors.
		for r_idx in range(k):
			A[r_idx,r_idx] += U_reg
		U[m, :] = np.linalg.solve(A, b)



	for n in prange(n_items):
		start_idx = item_indptr[n]
		end_idx = item_indptr[n+1]
		user_counter = end_idx - start_idx
		if user_counter == 0:
			item_bias[n] = 0
			continue

		numerator = 0.0
		for i in range(start_idx, end_idx):
			m = item_indices[i]
			r = item_data[i]

			dot_product = 0.0
			for j in range(k):
				dot_product += U[m,j] * V[n,j]
			numerator += lam * (r - global_average - user_bias[m] - dot_product)

		denominator = lam * user_counter + gamma
		item_bias[n] = numerator / denominator



	for n in prange(n_items):
		start_idx = item_indptr[n]
		end_idx = item_indptr[n+1]
		if start_idx == end_idx:
			V[n, :] = 0.0
			continue
		A = np.zeros((k, k), dtype=np.float32)
		b = np.zeros(k, dtype=np.float32)
		for i in range(start_idx, end_idx):
			m = item_indices[i]
			r = item_data[i]
			residual = (r - global_average - user_bias[m] - item_bias[n])
			for r_idx in range(k):
				U_r = U[m,r_idx]
				b[r_idx] += np.float32(lam * U_r * residual)
				for c_idx in range(k):
					A[r_idx,c_idx] += np.float32(lam * U_r * U[m,c_idx])
		# L2 regularization for item factors.
		for r_idx in range(k):
			A[r_idx,r_idx] += V_reg
		V[n, :] = np.linalg.solve(A, b)





In [ ]:
@njit(parallel=True)
def calculate_sse(
    user_indptr, user_data, user_indices,
    U, V, user_bias, item_bias, global_average
):
    n_users = len(user_indptr) - 1

    # Each parallel iteration writes to its own array element.
    user_sse = np.zeros(n_users, dtype=np.float64)
    user_counts = np.zeros(n_users, dtype=np.int64)

    k = U.shape[1]

    for m in prange(n_users):
        start_idx = user_indptr[m]
        end_idx = user_indptr[m + 1]

        local_sse = 0.0
        local_count = 0

        for i in range(start_idx, end_idx):
            n = user_indices[i]
            rating = user_data[i]

            prediction = (
                global_average
                + user_bias[m]
                + item_bias[n]
            )

            for j in range(k):
                prediction += U[m, j] * V[n, j]

            error = rating - prediction
            local_sse += error * error
            local_count += 1

        user_sse[m] = local_sse
        user_counts[m] = local_count

    total_sse = np.sum(user_sse)
    total_count = np.sum(user_counts)

    return total_sse, total_count


In [ ]:
# Train with k=2 for the embedding plot
lam = 0.01
gamma = 0.1
tau = 0.1
epochs = 30

user_bias = np.zeros(n_users, dtype=np.float32)
item_bias = np.zeros(n_items, dtype=np.float32)

k = 2
U_reg = 0.1
V_reg = 0.1
rng = np.random.default_rng(42)

U = rng.normal(0, 0.01, (n_users, k)).astype(np.float32)
V = rng.normal(0, 0.01, (n_items, k)).astype(np.float32)
I = np.eye(k, dtype=np.float32)

train_loss, test_loss = [], []
train_rmse, test_rmse = [], []


for epoch in range(epochs):
  als_algorithm(train_user_indptr, train_user_indices, train_user_data,
    train_item_indptr, train_item_indices, train_item_data,
    U, V, user_bias, item_bias, global_average,
    lam, gamma, tau, U_reg, V_reg, I, k)

  train_sse, train_count = calculate_sse(train_user_indptr, train_user_data, train_user_indices, U, V, user_bias, item_bias, global_average)
  test_sse, test_count = calculate_sse(test_user_indptr, test_user_data, test_user_indices, U, V, user_bias, item_bias, global_average)

  train_mse = train_sse / train_count
  test_mse = test_sse / test_count

  train_rmse_value = np.sqrt(train_mse)
  test_rmse_value = np.sqrt(test_mse)

  regularization = (
        gamma * np.sum(user_bias.astype(np.float64) ** 2)
        + tau * np.sum(item_bias.astype(np.float64) ** 2)
        + U_reg * np.sum(U.astype(np.float64) ** 2)
        + V_reg * np.sum(V.astype(np.float64) ** 2)
    )


  objective = lam * train_sse + regularization

  train_loss.append(train_mse)
  test_loss.append(test_mse)
  train_rmse.append(train_rmse_value)
  test_rmse.append(test_rmse_value)

  print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_mse:.4f} | Test Loss: {test_mse:.4f} | "
        f"Train RMSE: {train_rmse_value:.4f} | Test RMSE: {test_rmse_value:.4f} | "
        f"Objective: {objective:.4f}"
    )




In [ ]:
np.savez_compressed(
  "/content/drive/MyDrive/als_model_weights_k2.npz",
  U=U,
  V=V,
  user_bias=user_bias,
  item_bias=item_bias,
  global_average=global_average,
  unique_users=unique_users,
  unique_items=unique_items,
)

print("Model weights successfully saved to '/content/drive/MyDrive/als_model_weights_k2.npz'")


In [ ]:
weights= np.load("/content/drive/MyDrive/als_model_weights_k2.npz")

U = weights["U"]
V = weights["V"]
user_bias = weights["user_bias"]
item_bias = weights["item_bias"]
global_average = weights["global_average"]
unique_users = weights["unique_users"]
unique_items = weights["unique_items"]

user_to_index = {raw_id: index for index, raw_id in enumerate(unique_users)}
index_to_user = {index: raw_id for index, raw_id in enumerate(unique_users)}
item_to_index = {raw_id: index for index, raw_id in enumerate(unique_items)}
index_to_item = {index: raw_id for index, raw_id in enumerate(unique_items)}


In [ ]:
movies_df = pd.read_csv("/content/drive/MyDrive/movies.csv")

# Raw movie ID -> movie title
movie_title_by_id = dict(
    zip(movies_df["movieId"], movies_df["title"])
)

# Raw movie ID -> list of genres
genres_by_id = {
    row["movieId"]: row["genres"].split("|")
    for _, row in movies_df.iterrows()
}

# Keep only movies that have trained item vectors
index_to_movie_id = {
    index: int(movie_id)
    for index, movie_id in enumerate(unique_items)
}

movie_id_to_index = {
    movie_id: index
    for index, movie_id in index_to_movie_id.items()
}

print("Number of movies with trained vectors:", len(index_to_movie_id))
print(movie_title_by_id.get(1))

In [ ]:
def save_svg(filename):
    plt.savefig(filename + ".svg", format="svg", bbox_inches="tight")

In [ ]:
# Plot of trained 2D movie embeddings
if V.shape[1] != 2:
    raise ValueError("Load or train a model with k=2 first.")

movies_to_label = [
    "Toy Story (1995)",
    "Jumanji (1995)",
    "Toy Story 3 (2010)",
    "Babe (1995)",
    "Beauty and the Beast (1991)",
    "Babe: Pig in the City (1998)",
    "Incredibles, The (2004)",
    "Little Mermaid, The (1989)",
    "Pinocchio (1940)",
    "Finding Nemo (2003)",
    "Shrek (2001)",
    "Alien: Covenant - Prologue: The Crossing (2017)",
    "Elizabeth's Gift (2012)",
    "Shane Gillis: Beautiful Dogs (2023)",
    "Titanic (1997)",
    "The Rogues' Tavern (1936)"
]

plt.figure(figsize=(12, 9))

# Background of all movie vectors
plt.scatter(
    V[:, 0],
    V[:, 1],
    alpha=0.12,
    s=8,
    label="Other movies"
)

for title in movies_to_label:
    matching_ids = [
        movie_id
        for movie_id, movie_title in movie_title_by_id.items()
        if movie_title == title
    ]

    if not matching_ids:
        print("Title not found in metadata:", title)
        continue

    movie_id = matching_ids[0]

    if movie_id not in movie_id_to_index:
        print("No trained vector for:", title)
        continue

    idx = movie_id_to_index[movie_id]
    x, y = V[idx]

    plt.scatter(x, y, s=35)
    plt.annotate(
        title,
        (x, y),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.axhline(0, linestyle="--", alpha=0.4)
plt.axvline(0, linestyle="--", alpha=0.4)
plt.xlabel("Latent dimension 1")
plt.ylabel("Latent dimension 2")
plt.title("MovieLens: Trained 2D Movie Embeddings")
plt.legend()
plt.tight_layout()
save_svg("2D_Movie_embeddings")
plt.show()

In [ ]:
# Plot a genre embedding
genre_vectors = {}
genre_counts = {}

for genre in sorted({
    genre
    for genres in genres_by_id.values()
    for genre in genres
}):
    indices = [
        movie_id_to_index[movie_id]
        for movie_id, genres in genres_by_id.items()
        if genre in genres and movie_id in movie_id_to_index
    ]

    if indices:
        genre_vectors[genre] = np.mean(V[indices], axis=0)
        genre_counts[genre] = len(indices)

plt.figure(figsize=(11, 8))

for genre, vector in genre_vectors.items():
    x, y = vector
    plt.scatter(x, y, s=45)
    plt.annotate(
        genre,
        (x, y),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=9
    )

plt.axhline(0, linestyle="--", alpha=0.4)
plt.axvline(0, linestyle="--", alpha=0.4)
plt.xlabel("Latent dimension 1")
plt.ylabel("Latent dimension 2")
plt.title("Genre Centroids Derived from Trained Movie Embeddings")
plt.tight_layout()
save_svg("Genre_centroids")
plt.show()